# 📊 Análisis del Mercado Laboral en Ciencia de Datos — LATAM 2022–2024

**Autor:** [Tu Nombre]  
**Institución:** Duoc UC — Especialización en Ciencia de Datos  
**Fecha:** 2024  
**Dataset:** Mercado Laboral Data Science LATAM (1,200 registros | 5 países | 8 roles)

---

## 🎯 Objetivo del Proyecto

Explorar y analizar el mercado laboral de Ciencia de Datos en Chile y LATAM para identificar:
- ¿Qué roles pagan más y por qué?
- ¿Qué habilidades aumentan el sueldo?
- ¿Qué industrias son más atractivas?
- ¿Cómo ha evolucionado el mercado 2022–2024?
- ¿Vale la pena tener certificaciones cloud?

---

## 📋 Tabla de Contenidos
1. [Carga y exploración inicial](#1)
2. [Distribución salarial en Chile](#2)
3. [Sueldos por cargo y nivel](#3)
4. [Habilidades más demandadas](#4)
5. [Impacto de habilidades en el sueldo](#5)
6. [Análisis por industria](#6)
7. [Modalidad de trabajo](#7)
8. [Tendencia salarial 2022–2024](#8)
9. [Impacto de certificaciones cloud](#9)
10. [Comparación LATAM](#10)
11. [Heatmap industria × nivel](#11)
12. [Conclusiones e insights](#12)


## 1. Importación de librerías y carga de datos <a id='1'></a>

In [ ]:
# Importaciones
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
AZUL_OSCURO = '#1A237E'
AZUL_MEDIO  = '#3949AB'
AZUL_CLARO  = '#7986CB'
VERDE       = '#00897B'
NARANJA     = '#F57C00'
ROJO        = '#C62828'
GRIS        = '#455A64'
PALETA      = [AZUL_OSCURO, AZUL_MEDIO, AZUL_CLARO, VERDE, NARANJA, '#8E24AA', ROJO, GRIS]
FONDO       = '#F8F9FC'

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.facecolor': FONDO,
    'axes.facecolor':   FONDO,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

print("✅ Librerías cargadas correctamente")


In [ ]:
# Carga del dataset
df = pd.read_csv('../data/mercado_laboral_ds_latam.csv')

# Orden categórico para nivel de experiencia
ORDEN_NIVEL = ['Junior', 'Semi Senior', 'Senior', 'Lead']
df['nivel_experiencia'] = pd.Categorical(df['nivel_experiencia'],
                                          categories=ORDEN_NIVEL, ordered=True)

# Subconjunto Chile
df_chile = df[df['pais'] == 'Chile'].copy()
df_chile['nivel_experiencia'] = pd.Categorical(df_chile['nivel_experiencia'],
                                                categories=ORDEN_NIVEL, ordered=True)

print(f"📦 Dataset completo:  {len(df):,} registros × {len(df.columns)} columnas")
print(f"🇨🇱 Solo Chile:       {len(df_chile):,} registros")
print(f"🌎 Países:            {df['pais'].nunique()} ({', '.join(df['pais'].unique())})")
print(f"💼 Roles:             {df['cargo'].nunique()} tipos de cargo")
print(f"🏢 Industrias:        {df['industria'].nunique()}")
print(f"📅 Años:              {sorted(df['anio'].unique())}")


In [ ]:
# Vista inicial del dataset
df.head(5)


In [ ]:
# Información general
df.info()


In [ ]:
# Estadísticas descriptivas de sueldos (Chile)
desc = df_chile['sueldo_clp'].describe()
desc_fmt = desc.apply(lambda x: f'${x/1e6:.2f}M' if 'count' not in desc.name else x)
print("📊 Estadísticas salariales — Chile (en CLP):")
print(f"  N registros : {int(desc['count']):,}")
print(f"  Mínimo      : ${desc['min']/1e6:.2f}M")
print(f"  Mediana     : ${desc['50%']/1e6:.2f}M")
print(f"  Media       : ${desc['mean']/1e6:.2f}M")
print(f"  Máximo      : ${desc['max']/1e6:.2f}M")
print(f"  Desv. Est.  : ${desc['std']/1e6:.2f}M")


## 2. Distribución Salarial en Chile <a id='2'></a>

> Se analiza cómo se distribuyen los sueldos en el mercado de Data Science en Chile, comparando la mediana y media, y segmentando por nivel de experiencia.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de Sueldos — Data Science Chile (2022–2024)',
             fontsize=15, fontweight='bold', color=AZUL_OSCURO, y=1.02)

# Histograma
ax = axes[0]
ax.hist(df_chile['sueldo_clp']/1e6, bins=35, color=AZUL_MEDIO,
        edgecolor='white', linewidth=0.6, alpha=0.9)
mediana = df_chile['sueldo_clp'].median()/1e6
media   = df_chile['sueldo_clp'].mean()/1e6
ax.axvline(mediana, color=NARANJA, lw=2.2, linestyle='--', label=f'Mediana: ${mediana:.2f}M')
ax.axvline(media,   color=ROJO,    lw=2.2, linestyle=':',  label=f'Media:  ${media:.2f}M')
ax.set_xlabel('Sueldo (millones CLP)', fontsize=11)
ax.set_ylabel('Frecuencia', fontsize=11)
ax.set_title('Distribución salarial', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))

# Box plot por nivel
COLORS_BOX = [AZUL_CLARO, AZUL_MEDIO, AZUL_OSCURO, VERDE]
ax2 = axes[1]
data_box = [df_chile[df_chile['nivel_experiencia']==n]['sueldo_clp'].values/1e6
            for n in ORDEN_NIVEL]
bp = ax2.boxplot(data_box, patch_artist=True,
                 medianprops=dict(color=NARANJA, linewidth=2.5),
                 whiskerprops=dict(linewidth=1.4), capprops=dict(linewidth=1.4),
                 flierprops=dict(marker='o', markersize=4, alpha=0.4))
for patch, color in zip(bp['boxes'], COLORS_BOX):
    patch.set_facecolor(color); patch.set_alpha(0.85)
ax2.set_xticklabels(ORDEN_NIVEL, fontsize=10)
ax2.set_ylabel('Sueldo (millones CLP)', fontsize=11)
ax2.set_title('Rango salarial por nivel', fontsize=12, fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))

plt.tight_layout()
plt.show()


**🔍 Insights:**
- La distribución muestra una asimetría positiva: hay una cola de sueldos altos (Leads y Seniors en big tech/fintech).
- La **mediana es más representativa** que la media para el mercado general.
- Los Juniors tienen menor dispersión; conforme sube el nivel, la variabilidad aumenta significativamente.


## 3. Sueldo por Cargo y Nivel de Experiencia <a id='3'></a>

In [ ]:
# Tabla resumen
tabla = (df_chile.groupby(['cargo','nivel_experiencia'])['sueldo_clp']
         .median().unstack('nivel_experiencia').reindex(columns=ORDEN_NIVEL) / 1e6)
tabla = tabla.sort_values('Senior', ascending=False)
tabla.style.format('${:.2f}M').background_gradient(cmap='Blues', axis=None)


In [ ]:
pivot = tabla.copy()
COLORS_BOX = [AZUL_CLARO, AZUL_MEDIO, AZUL_OSCURO, VERDE]

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(pivot))
width = 0.20
for i, (nivel, color) in enumerate(zip(ORDEN_NIVEL, COLORS_BOX)):
    bars = ax.bar(x + i*width - width*1.5, pivot[nivel], width, label=nivel,
                  color=color, alpha=0.88, edgecolor='white', linewidth=0.5)
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.04, f'${h:.1f}M',
                    ha='center', va='bottom', fontsize=7.5, color=GRIS)

ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Sueldo Mediano (millones CLP)', fontsize=11)
ax.set_title('Sueldo Mediano por Cargo y Nivel — Chile',
             fontsize=13, fontweight='bold', color=AZUL_OSCURO)
ax.legend(title='Nivel', fontsize=10, title_fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))
plt.tight_layout()
plt.show()


**🔍 Insights:**
- **AI Engineer** y **MLOps Engineer** son los roles mejor pagados en todos los niveles, incluyendo Junior.
- Un **ML Engineer Senior** puede ganar más del **doble** que un **Data Analyst Senior**.
- Los roles de **BI Analyst** son los de menor remuneración, aunque son más accesibles a nivel de entrada.


## 4. Habilidades más Demandadas <a id='4'></a>

In [ ]:
from collections import Counter

todas_skills = []
for skills_str in df_chile['habilidades']:
    todas_skills.extend([s.strip() for s in skills_str.split(',')])
conteo = Counter(todas_skills)
top15 = pd.DataFrame(conteo.most_common(15), columns=['Habilidad','Menciones'])
print(top15.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colores_bar = [AZUL_OSCURO if i < 5 else AZUL_MEDIO if i < 10 else AZUL_CLARO
               for i in range(15)]
bars = ax.barh(top15['Habilidad'][::-1], top15['Menciones'][::-1],
               color=colores_bar[::-1], edgecolor='white', linewidth=0.5, alpha=0.9)
for bar, val in zip(bars, top15['Menciones'][::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9.5, color=GRIS, fontweight='bold')
ax.set_xlabel('N° de menciones en ofertas', fontsize=11)
ax.set_title('Top 15 Habilidades más Demandadas — Data Science Chile',
             fontsize=13, fontweight='bold', color=AZUL_OSCURO)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=AZUL_OSCURO, label='Top 5'),
                   Patch(facecolor=AZUL_MEDIO,  label='Top 6-10'),
                   Patch(facecolor=AZUL_CLARO,  label='Top 11-15')],
          loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()


**🔍 Insights:**
- **Python** y **SQL** son absolutamente no negociables — aparecen en prácticamente toda oferta de trabajo.
- Las **herramientas de visualización** (Power BI, Tableau) tienen mucha demanda, especialmente para perfiles Analyst/Scientist.
- Las **habilidades cloud** (Azure, AWS, GCP) aparecen con fuerza desde el nivel Semi Senior.


## 5. Impacto de Habilidades en el Sueldo <a id='5'></a>

> ¿Cuánto más gana un Data Scientist Senior **con** Azure versus **sin** Azure? Aquí medimos el impacto real.

In [ ]:
skills_clave = ['Azure','AWS','GCP','Docker','Spark','MLflow','Airflow',
                'TensorFlow','PyTorch','Deep Learning','NLP']
df_senior = df_chile[df_chile['nivel_experiencia'] == 'Senior']
resultados_sk = []
for sk in skills_clave:
    con = df_senior[df_senior['habilidades'].str.contains(sk,na=False)]['sueldo_clp'].median()
    sin = df_senior[~df_senior['habilidades'].str.contains(sk,na=False)]['sueldo_clp'].median()
    if pd.notna(con) and pd.notna(sin) and sin > 0:
        diff_pct = (con - sin) / sin * 100
        resultados_sk.append({'Habilidad': sk, 'Con skill (M CLP)': round(con/1e6,2),
                               'Sin skill (M CLP)': round(sin/1e6,2), 'Diferencia %': round(diff_pct,1)})

df_sk = pd.DataFrame(resultados_sk).sort_values('Diferencia %', ascending=False)
print(df_sk.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
colors_sk = [VERDE if v > 0 else ROJO for v in df_sk['Diferencia %']]
bars = ax.barh(df_sk['Habilidad'], df_sk['Diferencia %'],
               color=colors_sk, alpha=0.88, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, df_sk['Diferencia %']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'+{val:.1f}%', va='center', fontsize=10, color=GRIS, fontweight='bold')
ax.set_xlabel('Diferencia salarial vs. no tener la habilidad (%)', fontsize=11)
ax.set_title('Impacto de Habilidades en Sueldo — Data Scientists Senior Chile',
             fontsize=13, fontweight='bold', color=AZUL_OSCURO)
plt.tight_layout()
plt.show()


## 6. Análisis por Industria <a id='6'></a>

In [ ]:
ind_stats = (df_chile.groupby('industria')['sueldo_clp']
             .agg(['median','mean','count']).sort_values('median', ascending=False).reset_index())
ind_stats.columns = ['Industria','Mediana','Media','N']

fig, ax = plt.subplots(figsize=(12, 5))
colores_ind = [VERDE if v >= 2_500_000 else AZUL_MEDIO if v >= 1_800_000 else ROJO
               for v in ind_stats['Mediana']]
bars = ax.bar(ind_stats['Industria'], ind_stats['Mediana']/1e6,
              color=colores_ind, alpha=0.9, edgecolor='white', linewidth=0.5)
for bar, row in zip(bars, ind_stats.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.04,
            f'${row.Mediana/1e6:.2f}M\n(n={row.N})',
            ha='center', va='bottom', fontsize=8.5, color=GRIS)
ax.set_xticklabels(ind_stats['Industria'], rotation=30, ha='right', fontsize=9.5)
ax.set_ylabel('Sueldo Mediano (millones CLP)', fontsize=11)
ax.set_title('Sueldo Mediano por Industria — Chile',
             fontsize=13, fontweight='bold', color=AZUL_OSCURO)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=VERDE, label='Alto (+$2.5M)'),
                   Patch(facecolor=AZUL_MEDIO, label='Medio ($1.8-2.5M)'),
                   Patch(facecolor=ROJO, label='Bajo (<$1.8M)')], fontsize=9)
plt.tight_layout()
plt.show()


**🔍 Insights:**
- **Fintech/Banca** y **Tecnología** son los sectores que mejor pagan en Chile.
- **Gobierno/Público** y **Educación** están en el rango más bajo, pero ofrecen mayor estabilidad laboral.
- **Minería** y **Seguros** son sectores subestimados pero con buena remuneración, y menos competencia que tech.


## 7. Modalidad de Trabajo <a id='7'></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Modalidad de Trabajo — Chile', fontsize=14, fontweight='bold',
             color=AZUL_OSCURO, y=1.02)

mod_counts = df_chile['modalidad'].value_counts()
axes[0].pie(mod_counts.values, labels=mod_counts.index, autopct='%1.1f%%',
            colors=[AZUL_OSCURO, AZUL_MEDIO, AZUL_CLARO], startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=2),
            textprops=dict(fontsize=11))
axes[0].set_title('Distribución de modalidades', fontsize=11, fontweight='bold')

mod_sueldo = (df_chile.groupby('modalidad')['sueldo_clp']
              .median().reindex(['Presencial','Híbrido','Remoto']) / 1e6)
bars = axes[1].bar(mod_sueldo.index, mod_sueldo.values,
                   color=[AZUL_CLARO, AZUL_MEDIO, VERDE], alpha=0.9,
                   edgecolor='white', linewidth=0.5, width=0.5)
for bar, val in zip(bars, mod_sueldo.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
                 f'${val:.2f}M', ha='center', va='bottom', fontsize=11,
                 fontweight='bold', color=GRIS)
axes[1].set_ylabel('Sueldo Mediano (millones CLP)', fontsize=11)
axes[1].set_title('Sueldo mediano por modalidad', fontsize=11, fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))

pct = (mod_sueldo['Remoto'] - mod_sueldo['Presencial']) / mod_sueldo['Presencial'] * 100
axes[1].annotate(f'Remoto paga\n{pct:.0f}% más\nque presencial',
                 xy=(2, mod_sueldo['Remoto']), xytext=(1.5, mod_sueldo['Remoto']*0.80),
                 fontsize=9.5, color=VERDE, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=VERDE, lw=1.5))
plt.tight_layout()
plt.show()


## 8. Tendencia Salarial 2022–2024 <a id='8'></a>

In [ ]:
tend = (df_chile.groupby(['anio','nivel_experiencia'])['sueldo_clp']
        .median().unstack('nivel_experiencia') / 1e6)
COLORS_BOX = [AZUL_CLARO, AZUL_MEDIO, AZUL_OSCURO, VERDE]

fig, ax = plt.subplots(figsize=(10, 5))
for nivel, color in zip(ORDEN_NIVEL, COLORS_BOX):
    if nivel in tend.columns:
        ax.plot(tend.index, tend[nivel], marker='o', markersize=8,
                linewidth=2.5, color=color, label=nivel)
        for x, y in zip(tend.index, tend[nivel]):
            ax.annotate(f'${y:.2f}M', (x, y), textcoords='offset points',
                        xytext=(0, 10), ha='center', fontsize=9.5,
                        color=color, fontweight='bold')

ax.set_xticks([2022, 2023, 2024])
ax.set_ylabel('Sueldo Mediano (millones CLP)', fontsize=11)
ax.set_title('Evolución Salarial 2022–2024 por Nivel — Chile',
             fontsize=13, fontweight='bold', color=AZUL_OSCURO)
ax.legend(title='Nivel', fontsize=10, title_fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))
ax.set_xlim(2021.7, 2024.3)
plt.tight_layout()
plt.show()

# Calcular crecimiento
print("\n📈 Crecimiento salarial 2022 → 2024:")
for nivel in ORDEN_NIVEL:
    if nivel in tend.columns:
        crecimiento = (tend[nivel][2024] - tend[nivel][2022]) / tend[nivel][2022] * 100
        print(f"  {nivel:12s}: +{crecimiento:.1f}%")


## 9. Impacto de Certificaciones Cloud <a id='9'></a>

In [ ]:
from matplotlib.patches import Patch
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
COLORS_BOX = [AZUL_CLARO, AZUL_MEDIO, AZUL_OSCURO, VERDE]
fig.suptitle('Impacto de Certificaciones Cloud — Chile',
             fontsize=13, fontweight='bold', color=AZUL_OSCURO, y=1.02)

grupos, etiquetas, colores_bp = [], [], []
for nivel in ORDEN_NIVEL:
    sub = df_chile[df_chile['nivel_experiencia'] == nivel]
    grupos.extend([sub[sub['tiene_cert_cloud']==True]['sueldo_clp'].values/1e6,
                   sub[sub['tiene_cert_cloud']==False]['sueldo_clp'].values/1e6])
    etiquetas.extend([f'{nivel}\nCon cert.', f'{nivel}\nSin cert.'])
    colores_bp.extend([VERDE, ROJO])

bp2 = axes[0].boxplot(grupos, patch_artist=True,
                      medianprops=dict(color='white', linewidth=2.0),
                      whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2),
                      flierprops=dict(marker='o', markersize=3, alpha=0.4))
for patch, color in zip(bp2['boxes'], colores_bp):
    patch.set_facecolor(color); patch.set_alpha(0.75)
axes[0].set_xticklabels(etiquetas, fontsize=7.5)
axes[0].set_ylabel('Sueldo (millones CLP)', fontsize=10)
axes[0].set_title('Distribución por nivel', fontsize=11, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))
axes[0].legend(handles=[Patch(facecolor=VERDE, label='Con cert. cloud'),
                         Patch(facecolor=ROJO, label='Sin cert. cloud')], fontsize=9)

diffs = []
for nivel in ORDEN_NIVEL:
    sub = df_chile[df_chile['nivel_experiencia'] == nivel]
    con = sub[sub['tiene_cert_cloud']==True]['sueldo_clp'].median()
    sin = sub[sub['tiene_cert_cloud']==False]['sueldo_clp'].median()
    if pd.notna(con) and pd.notna(sin) and sin > 0:
        diffs.append({'Nivel': nivel, 'Diferencia %': (con-sin)/sin*100})

df_diffs = pd.DataFrame(diffs)
bars = axes[1].bar(df_diffs['Nivel'], df_diffs['Diferencia %'],
                   color=VERDE, alpha=0.88, edgecolor='white', linewidth=0.5, width=0.5)
for bar, val in zip(bars, df_diffs['Diferencia %']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'+{val:.1f}%', ha='center', va='bottom', fontsize=11,
                 fontweight='bold', color=VERDE)
axes[1].set_ylabel('Diferencia salarial (%)', fontsize=11)
axes[1].set_title('Bonus por tener cert. cloud', fontsize=11, fontweight='bold')
axes[1].set_ylim(0, df_diffs['Diferencia %'].max() * 1.25)
plt.tight_layout()
plt.show()


## 10. Comparación LATAM <a id='10'></a>

In [ ]:
paises_plot = ['Chile', 'México', 'Colombia', 'Argentina']
latam_data  = (df[df['pais'].isin(paises_plot)]
               .groupby(['pais','cargo'])['sueldo_clp']
               .median().unstack('cargo') / 1e6)
latam_data  = latam_data.reindex(paises_plot)

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(latam_data.columns))
width = 0.18
pal_paises = [AZUL_OSCURO, VERDE, NARANJA, ROJO]
for i, (pais_n, color) in enumerate(zip(paises_plot, pal_paises)):
    vals = latam_data.loc[pais_n].values
    ax.bar(x + i*width - width*1.5, vals, width, label=pais_n,
           color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(latam_data.columns, rotation=20, ha='right', fontsize=9.5)
ax.set_ylabel('Sueldo Mediano (millones CLP)', fontsize=11)
ax.set_title('Comparación de Sueldos LATAM por Cargo (CLP equivalente)',
             fontsize=13, fontweight='bold', color=AZUL_OSCURO)
ax.legend(title='País', fontsize=10, title_fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))
plt.tight_layout()
plt.show()


## 11. Heatmap Industria × Nivel <a id='11'></a>

In [ ]:
pivot_heat = (df_chile.groupby(['industria','nivel_experiencia'])['sueldo_clp']
              .median().unstack('nivel_experiencia')
              .reindex(columns=ORDEN_NIVEL) / 1e6)
pivot_heat  = pivot_heat.sort_values('Senior', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot_heat, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, linecolor='white', ax=ax,
            annot_kws={'size': 10, 'weight': 'bold'},
            cbar_kws={'label': 'Sueldo mediano (M CLP)'})
ax.set_xlabel('Nivel de Experiencia', fontsize=11)
ax.set_ylabel('Industria', fontsize=11)
ax.set_title('Heatmap: Sueldo Mediano (M CLP) — Industria × Nivel — Chile',
             fontsize=12, fontweight='bold', color=AZUL_OSCURO)
plt.tight_layout()
plt.show()


## 12. Conclusiones e Insights de Negocio <a id='12'></a>

### 🎯 Hallazgos principales

**1. El mercado salarial tiene alta variabilidad — el nivel importa más que el cargo**
- Pasar de Junior a Semi Senior puede significar entre $700K y $1.2M CLP de aumento mensual.
- Un Senior en cualquier área data supera con creces el sueldo promedio nacional.

**2. Los roles de IA y MLOps son los más valiosos del mercado**
- AI Engineer Junior parte en ~$2M CLP, por encima del mediano de Data Analyst Senior.
- MLOps Engineer es el perfil con mayor escasez y mejor compensado.

**3. Python + SQL son el piso mínimo — Cloud es el diferenciador**
- Habilidades cloud (Azure, AWS, GCP) generan un premium salarial de 12–19% en Senior.
- NLP y Deep Learning también impactan significativamente.

**4. El trabajo remoto sigue siendo un boost salarial relevante**
- El trabajo 100% remoto paga un ~26% más que presencial en el mercado chileno.
- Híbrido es ya la modalidad dominante (~45% del mercado).

**5. Fintech/Banca y Tecnología son las industrias destino**
- Minería y Seguros son nichos subestimados con buena remuneración y menos competencia.
- Educación y Gobierno son opciones válidas para ganar experiencia, no para maximizar ingreso.

**6. El mercado chileno sigue siendo el más competitivo de LATAM**
- Chile tiene los sueldos más altos en CLP equivalente para roles de datos, seguido de México.
- La tendencia 2022–2024 muestra un crecimiento de ~17% acumulado para roles Senior.

---

### 📌 Recomendaciones para un profesional junior en Chile

| Prioridad | Acción |
|-----------|--------|
| 🥇 Alta | Dominar Python + SQL (sin negociación) |
| 🥇 Alta | Obtener al menos 1 certificación cloud (Azure DP-900 es el inicio) |
| 🥈 Media | Especializarse en MLOps o NLP para acceder al rango alto desde Junior |
| 🥈 Media | Apuntar a Fintech/Banca o empresas tech nativas como primer empleo |
| 🥉 Baja | Negociar modalidad híbrida o remota desde el primer contrato |

---

*Dataset: Mercado Laboral Data Science LATAM 2022–2024 | 1,200 registros*  
*Metodología: Análisis exploratorio de datos (EDA) con Python, Pandas, Matplotlib y Seaborn*
